### Menghubungkan Google Drive
Cell kode di bawah ini digunakan untuk menghubungkan (mount) penyimpanan awan Google Drive ke dalam sesi Google Colab Anda menggunakan pustaka bawaan `google.colab`. Dengan melakukan ini, Anda dapat mengakses data yang tersimpan di Google Drive secara langsung atau menyimpan data hasil pemrosesan secara permanen tanpa khawatir kehilangan file saat sesi runtime Colab berakhir.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Menginstal Pustaka Faker
Cell kode di bawah ini berfungsi untuk menginstal pustaka `Faker` secara senyap (*quiet*) menggunakan manajer paket `pip`. Pustaka ini sangat berguna untuk membuat data tiruan (*synthetic/dummy data*) seperti nama, alamat, kota, dan data acak lainnya yang akan kita gunakan untuk menyusun dataset transaksi simulasi.

In [5]:
!pip install faker -q

### Mengimpor Pustaka (Library)
Cell kode di bawah ini digunakan untuk mengimpor seluruh pustaka Python yang dibutuhkan dalam proyek pengolahan data ini. Kita mengimpor `numpy` untuk komputasi numerik, `pandas` untuk manipulasi dan analisis data berbasis tabel (DataFrame), `Faker` untuk membuat data generator tiruan, serta pustaka bawaan `random` untuk memilih data acak.

In [6]:
import numpy as np
import pandas as pd
from faker import Faker
import random

### Membuat Dataset Transaksi Mentah
Cell kode di bawah ini menginisialisasi pembuatan dataset tiruan sebanyak 500 transaksi menggunakan generator acak dengan nilai *seed* 42 agar hasilnya selalu konsisten. Pada proses ini, kita sengaja menyuntikkan format harga yang berantakan, variasi format tanggal yang berbeda, teks yang tidak seragam, nilai kosong (*missing values*) pada beberapa kolom, serta menambahkan 15 baris data duplikat untuk menyimulasikan data kotor dunia nyata sebelum menyimpannya ke dalam file `transaksi_mentah.csv`.

In [7]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


### Menangani Nilai Kosong (Missing Values)
Cell kode di bawah ini mendeteksi jumlah nilai kosong (*null*) pada setiap kolom dataset, lalu melakukan pembersihan awal. Baris yang memiliki nilai kosong pada kolom nama pelanggan (`customer_name`) dan metode pembayaran (`payment_method`) akan dihapus, sedangkan nilai kosong pada kolom kota pengiriman (`shipping_city`) akan diisi dengan teks default "Tidak Diketahui" untuk menjaga keutuhan baris data lainnya.

In [8]:
print(df.isnull().sum())

df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


### Menghapus Baris Duplikat
Cell kode di bawah ini digunakan untuk mendeteksi dan menghapus data ganda (duplikat) di dalam dataset. Pertama, kode akan menampilkan jumlah data duplikat berdasarkan seluruh kolom dan berdasarkan ID transaksi, lalu menghapus baris duplikat tersebut menggunakan fungsi `drop_duplicates()` agar setiap transaksi unik hanya tercatat satu kali di dalam dataset.

In [9]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


### Menyamakan Format Teks (Standardisasi Teks)
Cell kode di bawah ini melakukan standardisasi pada kolom teks (`category`, `payment_method`, dan `shipping_city`) dengan cara menghapus spasi kosong yang tidak perlu di awal atau akhir kata serta mengubah tulisan menjadi format huruf besar di awal kata (*Title Case*). Selain itu, terdapat penyesuaian khusus untuk mengganti tulisan "Cod" menjadi singkatan standar "COD" agar semua nilai dalam kolom metode pembayaran konsisten.

In [10]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

### Membersihkan Kolom Harga
Cell kode di bawah ini mendefinisikan dan menerapkan fungsi kustom bernama `bersihkan_harga` untuk membersihkan kolom harga yang sebelumnya bertipe teks acak. Fungsi ini bekerja dengan cara menghapus simbol mata uang seperti "Rp", menghilangkan titik pemisah ribuan, mengubah koma desimal menjadi titik desimal, dan mengonversi format teks tersebut menjadi tipe angka desimal (*float*) agar siap untuk dianalisis dan dihitung.

In [11]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

### Menyamakan Format Tanggal
Cell kode di bawah ini membuat fungsi kustom bernama `parse_tanggal` untuk menangani variasi penulisan tanggal yang tidak konsisten di dalam dataset (seperti format tanda hubung, garis miring, atau format ISO). Fungsi ini mencoba mencocokkan setiap nilai tanggal dengan pola-pola format yang umum dan mengubah semuanya menjadi format tanggal standar internasional yang seragam yaitu `YYYY-MM-DD`.

In [12]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

### Memastikan Kesesuaian Tipe Data Akhir
Cell kode di bawah ini digunakan untuk menegaskan kembali tipe data pada kolom kuantitas (`quantity`) dan harga (`price`) setelah melalui proses pembersihan. Langkah ini penting untuk mengonversi kolom kuantitas secara tegas menjadi tipe bilangan bulat (*integer*) dan kolom harga menjadi bilangan pecahan (*float*) sehingga tidak ada kesalahan tipe data saat dilakukan perhitungan matematika di tahap berikutnya.

In [13]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

### Menyimpan Dataset Bersih Secara Lokal
Cell kode di bawah ini berfungsi untuk menyimpan DataFrame yang kini telah bersih sepenuhnya ke dalam format CSV dengan nama file `transaksi_bersih.csv` di penyimpanan lokal sementara Google Colab. Setelah proses ekspor selesai, kode akan menampilkan pesan konfirmasi beserta jumlah baris akhir yang berhasil disimpan, yaitu sebanyak 490 baris data yang siap digunakan.

In [14]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [15]:
import os

# Simpan DataFrame ke Google Drive dengan direktori yang dikoreksi
drive_dir_corrected = '/content/drive/MyDrive/BigData/Praktikum2'
drive_path_corrected = os.path.join(drive_dir_corrected, 'transaksi_bersih.csv')

# Buat direktori jika belum ada
os.makedirs(drive_dir_corrected, exist_ok=True)

df.to_csv(drive_path_corrected, index=False)
print(f"Dataset bersih berhasil disimpan di: {drive_path_corrected}")

Dataset bersih berhasil disimpan di: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih.csv
